[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nursnaaz/zero-to-genai-engineer/blob/main/11_LangGraph/notebooks/01_langgraph_fundamentals_and_agents.ipynb)

# LangGraph Fundamentals & Agents

Module 10 built `ProductionRAGChatbot` — a real, production-grade RAG pipeline. But look at
its `chat()` method: **retrieve → rerank → guardrail → generate, every single time, in that
exact order.** It cannot try again. It cannot decide, mid-run, to take a different path. If
the first retrieval is weak, the guardrail just refuses — there's no way back to "try a
better query."

That's not a limitation of RAG. It's a limitation of **chains**. A LangChain LCEL chain
(`prompt | llm | parser`) is a straight line: it runs forward, once, and stops. Anything that
needs a *loop* (retry), a *branch* (do X or Y depending on what the model just said), or a
*pause* (wait for a human) needs a different model of computation — a **graph**.

## What you'll build today

1. The smallest possible graph — two nodes, one edge
2. Conditional routing — branch on state, not just chain forward
3. A tool-calling agent, built by hand from a `StateGraph` (so you see what's underneath
   `create_agent`)
4. Memory — the same conversation, but now the framework remembers it for you
5. Streaming — watch the graph think, node by node and token by token

> **Latest API note (Aug 2026):** LangChain & LangGraph both shipped v1.0 this year.
> `create_agent` (in the `langchain` package) is now the recommended high-level agent
> builder — it's built *on top of* the exact `StateGraph` mechanics you're about to write by
> hand. `create_react_agent` (in `langgraph.prebuilt`) still works and you'll see it too, but
> treat it as the legacy prebuilt. Sources: [LangChain/LangGraph v1.0](https://www.langchain.com/blog/langchain-langgraph-1dot0),
> [create_react_agent reference](https://reference.langchain.com/python/langgraph.prebuilt/chat_agent_executor/create_react_agent).

## 0. Install dependencies

In [ ]:
%pip install -q langgraph>=0.6 langchain>=1.0 langchain-openai python-dotenv

import langgraph, langchain
print("langgraph:", langgraph.__version__)
print("langchain:", langchain.__version__)

## 1. Setup

Same pattern as every notebook in this course: load `OPENAI_API_KEY` from the `.env` file
(this time three directories up, since we're in `11_LangGraph/notebooks/`).

In [ ]:
import warnings, os
warnings.filterwarnings("ignore")

from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd().parent.parent / ".env")
print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 2. The smallest possible graph

Three ingredients, always:

- **State** — a typed dict that flows through the graph. Every node reads from it and returns
  a partial update.
- **Nodes** — plain Python functions: `(state) -> dict`.
- **Edges** — how nodes connect. `START` and `END` are sentinel nodes marking entry and exit.

No LLM yet — just the mechanics.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


class JokeState(TypedDict):
    topic: str
    joke: str


def pick_topic(state: JokeState) -> dict:
    return {"topic": "stateful graphs"}


def write_joke(state: JokeState) -> dict:
    joke = llm.invoke(f"Write a one-line joke about {state['topic']}.").content
    return {"joke": joke}


builder = StateGraph(JokeState)
builder.add_node("pick_topic", pick_topic)
builder.add_node("write_joke", write_joke)
builder.add_edge(START, "pick_topic")
builder.add_edge("pick_topic", "write_joke")
builder.add_edge("write_joke", END)

joke_graph = builder.compile()
result = joke_graph.invoke({})
print(result)

**Always look at the graph, not just the code.** `draw_mermaid()` renders the actual compiled
graph — this should become a reflex every time you build one.

In [ ]:
print(joke_graph.get_graph().draw_mermaid())

# In a notebook that supports it, you can also render this as an image:
try:
    from IPython.display import Image, display
    display(Image(joke_graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print("(Mermaid PNG rendering needs internet access to the mermaid.ink API; "
          "the text diagram above is the source of truth.)", e)

## 3. Conditional edges — branching on state

A chain can't decide *which* step comes next. A graph can: `add_conditional_edges` takes a
router function that inspects state and returns the name of the next node.

In [ ]:
class RouteState(TypedDict):
    question: str
    complexity: str
    answer: str


def classify(state: RouteState) -> dict:
    verdict = llm.invoke(
        f"Classify this question as exactly one word, SIMPLE or COMPLEX, nothing else:\n{state['question']}"
    ).content.strip().upper()
    return {"complexity": "COMPLEX" if "COMPLEX" in verdict else "SIMPLE"}


def short_answer(state: RouteState) -> dict:
    return {"answer": llm.invoke(f"Answer in ONE sentence: {state['question']}").content.strip()}


def detailed_answer(state: RouteState) -> dict:
    return {"answer": llm.invoke(f"Answer thoroughly, with an example: {state['question']}").content.strip()}


def route(state: RouteState) -> str:
    return "detailed_answer" if state["complexity"] == "COMPLEX" else "short_answer"


router_builder = StateGraph(RouteState)
router_builder.add_node("classify", classify)
router_builder.add_node("short_answer", short_answer)
router_builder.add_node("detailed_answer", detailed_answer)
router_builder.add_edge(START, "classify")
router_builder.add_conditional_edges(
    "classify", route, {"short_answer": "short_answer", "detailed_answer": "detailed_answer"}
)
router_builder.add_edge("short_answer", END)
router_builder.add_edge("detailed_answer", END)
router_graph = router_builder.compile()

for q in ["What's 2 + 2?", "Explain why transformers use self-attention instead of recurrence."]:
    out = router_graph.invoke({"question": q})
    print(f"[{out['complexity']}] {q}\n  -> {out['answer'][:160]}...\n")

## 4. A tool-calling agent — built by hand first

`create_agent` and `create_react_agent` both hide the same four pieces: a model node that can
call tools, a tools node that executes them, and a conditional edge that loops between them
until the model stops asking for tools. Build that loop by hand once, so the prebuilt helpers
stop being a black box.

In [ ]:
from langchain_core.tools import tool
from langgraph.graph import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition


@tool
def get_weather(city: str) -> str:
    """Return the current weather for a city (simulated for the demo)."""
    fake = {"dubai": "38C, clear", "london": "16C, light rain", "chennai": "31C, humid"}
    return fake.get(city.lower(), f"{city}: 27C, partly cloudy")


@tool
def word_count(text: str) -> int:
    """Count the number of words in a string."""
    return len(text.split())


tools = [get_weather, word_count]
llm_with_tools = llm.bind_tools(tools)


def call_model(state: MessagesState) -> dict:
    return {"messages": [llm_with_tools.invoke(state["messages"])]}


agent_builder = StateGraph(MessagesState)
agent_builder.add_node("agent", call_model)
agent_builder.add_node("tools", ToolNode(tools))
agent_builder.add_edge(START, "agent")
agent_builder.add_conditional_edges("agent", tools_condition)   # -> "tools" or END, based on tool_calls
agent_builder.add_edge("tools", "agent")

hand_rolled_agent = agent_builder.compile()

for chunk in hand_rolled_agent.stream(
    {"messages": [("user", "What's the weather in Dubai, and how many words are in "
                            "'stateful agent graphs beat linear chains'?")]},
    stream_mode="values",
):
    chunk["messages"][-1].pretty_print()

### The same agent, the fast way

Now that you've seen what's underneath, here's the one-liner. `create_agent` (LangChain 1.0)
is preferred for new code; `create_react_agent` (legacy `langgraph.prebuilt`) is shown as a
fallback for pinned older environments — both compile to essentially the graph you just built.

In [ ]:
try:
    from langchain.agents import create_agent
    fast_agent = create_agent(model="gpt-4o-mini", tools=tools)
    print("Using langchain.agents.create_agent (v1.0+)")
except ImportError:
    from langgraph.prebuilt import create_react_agent
    fast_agent = create_react_agent(llm, tools)
    print("langchain.agents.create_agent not available -- falling back to "
          "langgraph.prebuilt.create_react_agent (legacy)")

result = fast_agent.invoke({"messages": [("user", "What's the weather in London?")]})
result["messages"][-1].pretty_print()

## 5. Memory — conversations that persist across calls

In Module 10, `ProductionRAGChatbot` stored history in `self.history`, a plain Python list —
memory that dies the moment the process exits. A **checkpointer** does the same job, but as a
first-class part of the graph: it snapshots state after every node, keyed by a `thread_id`.

Compile the *same* graph with a checkpointer, then invoke it twice under the same
`thread_id` — no code changes to the nodes themselves.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()
memory_agent = agent_builder.compile(checkpointer=checkpointer)

config_a = {"configurable": {"thread_id": "student-1"}}
memory_agent.invoke({"messages": [("user", "My name is Noor. Remember that.")]}, config_a)
follow_up = memory_agent.invoke({"messages": [("user", "What's my name?")]}, config_a)
print("Same thread  :", follow_up["messages"][-1].content)

config_b = {"configurable": {"thread_id": "student-2"}}
isolated = memory_agent.invoke({"messages": [("user", "What's my name?")]}, config_b)
print("Other thread :", isolated["messages"][-1].content)

`InMemorySaver` disappears when the kernel restarts — swap in `SqliteSaver` or
`PostgresSaver` (same `langgraph.checkpoint.*` interface) and every conversation survives a
restart with **zero changes to the graph itself**. That portability is the point: memory is a
property of *how you compile*, not something you hand-code into every node.

## 6. Streaming — three different views of the same run

- `stream_mode="values"` — the full state after each node (what you saw above)
- `stream_mode="updates"` — only the diff each node produced (good for a live trace/debug panel)
- `stream_mode="messages"` — individual LLM tokens as they're generated (good for a chat UI)

In [ ]:
print("--- updates ---")
for update in memory_agent.stream(
    {"messages": [("user", "In one sentence, why do graphs beat chains for agents?")]},
    config_a, stream_mode="updates",
):
    for node_name, node_output in update.items():
        print(f"[{node_name}] ->", {k: (v if k != 'messages' else '<messages>') for k, v in node_output.items()})

print("\n--- messages (token streaming) ---")
for token_chunk, metadata in memory_agent.stream(
    {"messages": [("user", "Name one real-world use case for a multi-agent supervisor, in five words.")]},
    config_a, stream_mode="messages",
):
    if token_chunk.content:
        print(token_chunk.content, end="", flush=True)
print()

## Summary

| Chain (Module 10 style) | Graph (LangGraph) |
|---|---|
| Fixed, linear order | Any order, including loops back to an earlier node |
| Can't branch on the model's own output | `add_conditional_edges` routes on live state |
| Memory = a list you manage yourself | Memory = `checkpointer` + `thread_id`, framework-managed |
| All-or-nothing execution | `stream_mode` exposes per-node and per-token progress |
| No pause point | (next notebook) `interrupt()` pauses and resumes mid-graph |

**Next: `02_human_in_the_loop_and_multi_agent.ipynb`** — pausing a graph for human approval,
long-term memory that survives across different conversations, and a supervisor coordinating
multiple specialist agents.